# FIXED POINT

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import copy

## Classes

In [2]:
class FixedPoint:
    def __init__(self, fl: int, value: int):
        """
        初始化 FixedPoint 類別實例。
        :param fl: Fractional length (小數部分的長度)，必須是整數
        :param value: 固定小數的實際值，必須是整數
        """
        if not isinstance(fl, int):
            raise TypeError(f"fl need to be integer {type(fl).__name__}")
        if not isinstance(value, int):
            raise TypeError(f"value need to be integer {type(value).__name__}")
        if fl < 0:
            raise ValueError("fl need to be nonnegative")

        self.fl = fl
        self.value = value
    def show_detail(self):
        print(f"fl: {self.fl}, decimal_value: {self.value}, true_fixed_point: {self.value*2**(-self.fl)}, ")
    def __int__(self):
        return self.value // (2**self.fl)
    def __add__(self, other):
        if not isinstance(other, FixedPoint):
            raise ValueError("add wrong")
            return NotImplemented
        # 使用兩者的 max fractional length 作為結果的 fl
        max_fl = max(self.fl, other.fl)
        # 將 self 和 other 都轉換至相同的 fractional length 再相加
        val = (self.value * (2**(max_fl - self.fl))) + (other.value * (2**(max_fl - other.fl)))
        return FixedPoint(fl=max_fl, value=val)

    def __sub__(self, other):
        if not isinstance(other, FixedPoint):
            raise ValueError("sub wrong")
            return NotImplemented
        # 使用兩者的 max fractional length 作為結果的 fl
        max_fl = max(self.fl, other.fl)
        # 將 self 和 other 都轉換至相同的 fractional length 再相減
        val = (self.value * (2**(max_fl - self.fl))) - (other.value * (2**(max_fl - other.fl)))
        return FixedPoint(fl=max_fl, value=val)

    def __mul__(self, other):
        if not isinstance(other, FixedPoint):
            raise ValueError("mul wrong")
            return NotImplemented
        # 乘法時 fl 相加
        new_fl = self.fl + other.fl
        new_val = self.value * other.value
        return FixedPoint(fl=new_fl, value=new_val)

    def __repr__(self):
        return f"FixedPoint(fl={self.fl}, value={self.value}, { self.value // (2**self.fl):b}.{self.value - (self.value // (2**self.fl) )*2**self.fl:0{self.fl}b}, decimal_value={self.value*2**(-self.fl)})"

    def __str__(self):
        return f"FixedPoint(fl={self.fl}, value={self.value}, { self.value // (2**self.fl):b}.{self.value - (self.value // (2**self.fl) )*2**self.fl:0{self.fl}b}, decimal_value={self.value*2**(-self.fl)})"

    def FloatingPointVal(self):
        return self.value*(2**(-self.fl))


In [3]:
def quantized_fl(A: FixedPoint, after_quantized_fl: int):
  if( after_quantized_fl >= A.fl):
    raise ValueError("after_quantized_fl need to smaller than original fl")

  after = FixedPoint(fl=0,value=0)
  after.fl = after_quantized_fl
  after.value = ( A.value+2**(A.fl-after_quantized_fl-1) ) // 2**(A.fl-after_quantized_fl)
  return after

In [4]:
def FloatingPointToFixedPoint(A: float, fl: int):
  after = FixedPoint(fl=0,value=0)
  after.fl = fl
  after.value = int(A*2**(fl+1)+1) // 2
  return after

In [5]:
def int_to_signed_binary_str(value: int, bits: int) -> str:
    """
    將整數 value 轉換為指定 bits 位元的二補數 (signed binary) 字串表示。
    假設 bits >= 所需表示該整數的最小位元數。
    """
    # 檢查位元數是否足夠
    if bits <= 0:
        raise ValueError("bits 必須大於 0")

    # 檢查數值是否在該位元數的表示範圍內
    # 範圍： -2^(bits-1) ~ 2^(bits-1)-1
    if value < -(1 << (bits - 1)) or value > (1 << (bits - 1)) - 1:
        raise ValueError(f"數值 {value} 無法以 {bits} 位元表示為有號二進位。")

    # 若為負數，轉二補數時先將 value 加上 2^bits
    if value < 0:
        value = (1 << bits) + value

    # 將整數轉為二進位字串並補零
    binary_str = format(value, 'b').zfill(bits)
    return binary_str

In [6]:
class Mesh:
    def __init__(self):
        self.vertices = np.array([[]])
        self.faces = np.array([[]])
        self.colors = np.array([[]])  # Optional: To store vertex colors

        #calculated results
        self.clip_space_vertices = np.array([[]])
        self.NDC_vertices = np.array([[]])
        self.screen_space_vertices = np.array([[]])

    def load_obj(self, filepath):
        """
        Load a .obj file and extract vertex positions.
        filepath: Path to the .obj file
        Returns: Nx3 numpy array of vertex positions
        """
        vertices = []
        faces = []
        with open(filepath, 'r') as file:
            for line in file:
                if line.startswith('v '):  # Line defines a vertex
                    parts = line.strip().split()
                    if len(parts) >= 4:
                        vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
                elif line.startswith('f '):  # Line defines a face
                    parts = line.strip().split()
                    # OBJ indices are 1-based, so we subtract 1 for 0-based indexing
                    face = [int(part.split('/')[0]) - 1 for part in parts[1:4]]
                    faces.append(face)
        self.vertices = np.array(vertices)
        self.faces = np.array(faces)

    def load_ply(self, filepath):
        """
        Load a .ply file and extract vertex positions and face indices.
        Assumes the PLY file is in ASCII format with properties: x, y, z, red, green, blue
        and faces defined as a list of vertex indices.

        filepath: Path to the .ply file
        """
        with open(filepath, 'r') as file:
            line = file.readline().strip()
            if line != 'ply':
                raise ValueError("The file does not start with 'ply' header.")

            # Initialize variables
            num_vertices = 0
            num_faces = 0
            header_ended = False
            properties = []

            # Parse header
            while not header_ended:
                line = file.readline().strip()
                if line.startswith('element vertex'):
                    num_vertices = int(line.split()[-1])
                elif line.startswith('element face'):
                    num_faces = int(line.split()[-1])
                elif line.startswith('property'):
                    properties.append(line)
                elif line == 'end_header':
                    header_ended = True

            # Determine property indices (optional: if you want to store colors)
            property_names = [prop.split()[-1] for prop in properties]
            try:
                x_idx = property_names.index('x')
                y_idx = property_names.index('y')
                z_idx = property_names.index('z')
            except ValueError:
                raise ValueError("PLY file does not contain x, y, z properties.")

            # Optional: Check for color properties
            has_color = all(color in property_names for color in ['red', 'green', 'blue'])
            if has_color:
                red_idx = property_names.index('red')
                green_idx = property_names.index('green')
                blue_idx = property_names.index('blue')

            # Read vertex data
            vertices = []
            colors = []
            for _ in range(num_vertices):
                parts = file.readline().strip().split()
                if len(parts) < 3:
                    raise ValueError("Vertex line does not have enough coordinates.")
                xtemp = FloatingPointToFixedPoint(A=float(parts[x_idx]),fl=20)
                ytemp = FloatingPointToFixedPoint(A=float(parts[y_idx]),fl=20)
                ztemp = FloatingPointToFixedPoint(A=float(parts[z_idx]),fl=20)
                vertex = [xtemp, ytemp, ztemp]
                vertices.append(vertex)
                if has_color:
                    Rtemp = FixedPoint(fl=0, value=int(parts[red_idx]))
                    Gtemp = FixedPoint(fl=0, value=int(parts[green_idx]))
                    Btemp = FixedPoint(fl=0, value=int(parts[blue_idx]))
                    color = [Rtemp, Gtemp, Btemp]
                    colors.append(color)

            self.vertices = np.array(vertices)
            if has_color:
                self.colors = np.array(colors)
            else:
                self.colors = None  # Or handle as needed

            # Read face data
            faces = []
            for _ in range(num_faces):
                parts = file.readline().strip().split()
                if len(parts) < 4:
                    raise ValueError("Face line does not have enough indices.")
                vertex_count = int(parts[0])
                if vertex_count != 3:
                    raise ValueError("Only triangular faces are supported.")
                # PLY indices are 0-based
                face = [int(idx) for idx in parts[1:4]]
                faces.append(face)

            self.faces = np.array(faces)

In [7]:
class Camera:
    def __init__(self, eyeX: float,   eyeY: float,   eyeZ: float,
              centerX: float,  centerY: float, centerZ: float,
              upX: float,    upY: float,   upZ: float,
              screen_W: int, screen_H: int,
              near_clipping_plane: float, far_clipping_plane: float,
              fov: float):
        self.eyeX = FloatingPointToFixedPoint(A=eyeX, fl=20)
        self.eyeY = FloatingPointToFixedPoint(A=eyeY, fl=20)
        self.eyeZ = FloatingPointToFixedPoint(A=eyeZ, fl=20)
        self.eye = np.array([self.eyeX, self.eyeY, self.eyeZ])
        self.centerX = FloatingPointToFixedPoint(A=centerX, fl=20)
        self.centerY = FloatingPointToFixedPoint(A=centerY, fl=20)
        self.centerZ = FloatingPointToFixedPoint(A=centerZ, fl=20)
        self.center = np.array([self.centerX, self.centerY, self.centerZ])
        self.upX = FloatingPointToFixedPoint(A=upX, fl=20)
        self.upY = FloatingPointToFixedPoint(A=upY, fl=20)
        self.upZ = FloatingPointToFixedPoint(A=upZ, fl=20)
        self.up = np.array([self.upX, self.upY, self.upZ])

        self.screen_W = screen_W
        self.screen_H = screen_H
        self.aspect_ratio = screen_W / screen_H

        self.screen_buffer = np.zeros((screen_H, screen_W, 3), dtype=np.uint8)
        self.screen_buffer.fill(255)
        #self.screen_depth_buffer = np.zeros((screen_H, screen_W))
        #self.screen_depth_buffer.fill(1)

        #self.screen_buffer = [ FixedPoint(fl=0, value=255) for C in range(self.screen_H*self.screen_W) ]
        self.screen_depth_buffer = [ FixedPoint(fl=20, value=1048575) for D in range(self.screen_H*self.screen_W)]

        self.near_clipping_plane = near_clipping_plane
        self.far_clipping_plane = far_clipping_plane
        self.fov = fov
    def set_eye(self, eyeX: float, eyeY: float, eyeZ: float):
        self.eyeX = FloatingPointToFixedPoint(A=eyeX, fl=20)
        self.eyeY = FloatingPointToFixedPoint(A=eyeY, fl=20)
        self.eyeZ = FloatingPointToFixedPoint(A=eyeZ, fl=20)
        self.eye = np.array([self.eyeX, self.eyeY, self.eyeZ])
    def draw_screen(self):
        plt.imshow(self.screen_buffer)
        plt.axis('off')
        plt.show()

### Projection Matrix

The matrix is precalculated and hard-wired in the verilog code.

In [8]:
def calc_Projection_matrix( camera: Camera):

  fy = 1.0 / np.tan( np.radians( camera.fov ) / 2 )
  fx = fy / camera.aspect_ratio

  far = camera.far_clipping_plane
  near = camera.near_clipping_plane
  Projection = [ [ fx,  0,       0       ,       0        ],
          [ 0,  fy,       0       ,       0        ],
          [ 0,  0, - (far+near) / (far-near), (-2*far*near) / (far-near) ],
          [ 0,  0,       -1       ,       0        ] ]

  for row in range(4):
    for col in range(4):
      Projection[row][col] = FloatingPointToFixedPoint(A=Projection[row][col], fl=21)
  return Projection

## View marix

### Inverse square

In [9]:
def inv_sqrt_LUT( C: int ):
  c = (1/np.sqrt(C))
  return FloatingPointToFixedPoint(A=c, fl=24)

#### Create LUT

In [10]:
# print('\tOUT = 0;')
# print(f'\tcase(IN):')
# for i in range( 3, (2**7)*3 ):
#     ans = inv_sqrt_LUT( i )
#     print(f'\t\t{i}: OUT = {ans.value};')
# print('\tendcase')

#### Newton's method

In [11]:
def inv_sqrt( x: FixedPoint, y:FixedPoint, z:FixedPoint ):
  x_2 = quantized_fl(x*x, after_quantized_fl=24)
  y_2 = quantized_fl(y*y, after_quantized_fl=24)
  z_2 = quantized_fl(z*z, after_quantized_fl=24)

  SUM = x_2 + y_2 + z_2
  # print(x, y, z, SUM)

  X0 = inv_sqrt_LUT( int(SUM) )
  # iteration 1
  X0_2 = quantized_fl(X0*X0, after_quantized_fl=24)

  coeff_0 = FloatingPointToFixedPoint(A=3.0, fl=48) - (SUM * X0_2)
  coeff_0.fl += 1
  coeff_0 = quantized_fl(A=coeff_0, after_quantized_fl=24)

  X1 = quantized_fl(A=X0 * coeff_0, after_quantized_fl=24)
  # iteration 2
  X1_2 = quantized_fl(X1*X1, after_quantized_fl=24)

  coeff_1 = FloatingPointToFixedPoint(A=3.0, fl=48) - (SUM * X1_2)
  coeff_1.fl += 1
  coeff_1 = quantized_fl(A=coeff_1, after_quantized_fl=24)

  X2 = quantized_fl(A=X1 * coeff_1, after_quantized_fl=24)
  # iteration 3
  X2_2 = quantized_fl(X2*X2, after_quantized_fl=24)

  coeff_2 = FloatingPointToFixedPoint(A=3.0, fl=48) - (SUM * X2_2)
  coeff_2.fl += 1
  coeff_2 = quantized_fl(A=coeff_2, after_quantized_fl=24)

  X3 = quantized_fl(A=X2 * coeff_2, after_quantized_fl=23)
  X3 = X3 if (X3.FloatingPointVal()>=0) else FixedPoint(fl=X3.fl, value=0)-X3

  return X3

### Cross product

In [12]:
def cross( U, V ):

  #Ux, Uy, Uz, Vx, Vy, Vz
  Ux = U[0]
  Uy = U[1]
  Uz = U[2]
  Vx = V[0]
  Vy = V[1]
  Vz = V[2]

  UyVz = quantized_fl(A=Uy*Vz, after_quantized_fl=25)
  UzVx = quantized_fl(A=Uz*Vx, after_quantized_fl=25)
  UxVy = quantized_fl(A=Ux*Vy, after_quantized_fl=25)

  UzVy = quantized_fl(A=Uz*Vy, after_quantized_fl=25)
  UxVz = quantized_fl(A=Ux*Vz, after_quantized_fl=25)
  UyVx = quantized_fl(A=Uy*Vx, after_quantized_fl=25)

  outx = quantized_fl(A=UyVz - UzVy, after_quantized_fl=24)
  outy = quantized_fl(A=UzVx - UxVz, after_quantized_fl=24)
  outz = quantized_fl(A=UxVy - UyVx, after_quantized_fl=24)

  return np.array([outx, outy, outz])

### Negative Dot product

In [13]:
def neg_dot( U, V ):
  unit_x = U[0]
  unit_y = U[1]
  unit_z = U[2]
  x2 = V[0]
  y2 = V[1]
  z2 = V[2]

  assert unit_x.fl == 24

  product_x = quantized_fl(A=unit_x*x2, after_quantized_fl=24)
  product_y = quantized_fl(A=unit_y*y2, after_quantized_fl=24)
  product_z = quantized_fl(A=unit_z*z2, after_quantized_fl=24)

  out = product_x + product_y + product_z
  out = FloatingPointToFixedPoint(A=0.0, fl=24) - out
  out = quantized_fl(A=out, after_quantized_fl=17)

  return out

## PACK vertex_shader together

vertex_shader:  
input:
- `camera`: the current camera
- `vertices_in_face`: [ vertex1, vertex2, vertex3 ]

output:  
Returns the x, y and depth of the input vertices in the same order.
- [ [screen_x, screen_y, depth],  
   [screen_x, screen_y, depth],  
   [screen_x, screen_y, depth] ]

In [14]:
def vertex_shader(camera: Camera, vertices_in_face: np.array):
  projection = calc_Projection_matrix(camera)
  # view matrix
  # Get CamZ
  CamZ = camera.eye - camera.center
  CamZ = CamZ * inv_sqrt(CamZ[0],CamZ[1],CamZ[2])
  for i in range(3):
    CamZ[i] = quantized_fl(A=CamZ[i], after_quantized_fl=24)

  # Get CamX
  CamX = cross(camera.up, CamZ)
  # Normalize
  tmpX = copy.deepcopy(CamX)
  tmpX[0].fl = 20
  tmpX[0].value = tmpX[0].value // 4
  tmpX[1].fl = 20
  tmpX[1].value = tmpX[1].value // 4
  tmpX[2].fl = 20
  tmpX[2].value = tmpX[2].value // 4
  tmp = inv_sqrt( tmpX[0], tmpX[1], tmpX[2] )
  tmp.fl = 21

  for i in range(3):
    CamX[i] = quantized_fl(A=CamX[i] * tmp, after_quantized_fl=24)

  # Get CamY
  tmpZ = copy.deepcopy(CamZ)
  for i in range(3):
    tmpZ[i].fl = 20
    tmpZ[i].value = tmpZ[i].value // 16
  CamY = cross(tmpZ, CamX)

  view = [ [ CamX[0],     CamY[0], CamZ[0], neg_dot(CamX, camera.eye)],
        [ CamX[1],     CamY[1], CamZ[1], neg_dot(CamY, camera.eye)],
        [ CamX[2],     CamY[2], CamZ[2], neg_dot(CamZ, camera.eye)],
        [ FixedPoint(17,0), FixedPoint(17,0), FixedPoint(17,0),          FixedPoint(17,1*2**17)         ] ]

  # GET MVP
  MVP = []
  for i in range(4):
    MVP.append( [ FixedPoint(0,0) for j in range(4) ] )
  MVP_sum = copy.deepcopy(MVP)

  for cnt in range(4):
    product = []
    for i in range(4):
      product.append( [ FixedPoint(0,0) for j in range(4) ] )
    product_quant = copy.deepcopy(product)

    for col in range(4):
      for row in range(4):

        product[row][col] = projection[cnt][row] * view[row][col]
        product_quant[row][col] = quantized_fl(product[row][col], 14)

    for col in range(4):
      MVP_sum[cnt][col] = (product_quant[0][col] + product_quant[1][col] + product_quant[2][col] + product_quant[3][col]);
      MVP[cnt][col] = quantized_fl(A=MVP_sum[cnt][col], after_quantized_fl=12)
  MVP = np.array(MVP)

  # Transformation
  res = []
  for vertex in vertices_in_face:
    vertex = np.append(vertex, FixedPoint(fl=0, value=1))

    for col in range(4):
      for row in range(4):
        product[row][col] = vertex[row] * MVP.T[row][col]
        product[row][col] = quantized_fl(product[row][col], 10)
        # print(f'product[{row}][{col}]({product[row][col].FloatingPointVal()}) = vertex[{row}]({vertex[row].FloatingPointVal()}) * MVP.T[{row}][{col}]({MVP.T[row][col].FloatingPointVal()})')
    # print(np.array(product))

    SUM = [ FixedPoint(0,0) for i in range(4) ]
    for col in range(4):
      SUM[col] = product[0][col] + product[1][col] + product[2][col] + product[3][col]
      SUM[col] = quantized_fl(SUM[col], 8)
      # print(f'SUM[{col}] = {SUM[col].FloatingPointVal()}')
    SUM = np.array(SUM)
    # for j in range(4):
    #   binary = int_to_signed_binary_str(SUM[j].value, 24)
    #   print( f"{int(binary, 2)}", end=" " )
    # print()
    for i in range(3):
      if( (SUM[i].value < 0) ^ (SUM[3].value < 0) ):
        if(abs(SUM[i].value) >= abs(SUM[3].value)):
          SUM[i].value = -(2**12)
          SUM[i].fl = 12
        else:
          SUM[i].value = -int(abs(SUM[i].value) * 2**12 // abs(SUM[3].value))
          SUM[i].fl = 12
      else:
        if(abs(SUM[i].value) >= abs(SUM[3].value)):
          SUM[i].value = 2**12
          SUM[i].fl = 12
        else:
          SUM[i].value = int(abs(SUM[i].value) * 2**12 // abs(SUM[3].value))
          SUM[i].fl = 12
    # print("NDC:")
    # for j in range(4):
    #   binary = int_to_signed_binary_str(SUM[j].value, 24)
    #   print( f"{int(binary, 2)}", end=" " )
    # print()
    SUM[2].value *= 2**7
    SUM[2].fl = 20
    depth = SUM[2]

    screen_x = SUM[0]+FloatingPointToFixedPoint(A=1.0, fl=12)
    screen_x.fl = 13
    screen_x = screen_x * FloatingPointToFixedPoint(A=1280, fl=0)
    screen_x = quantized_fl(screen_x, 0)

    screen_y = SUM[1]
    screen_y.fl = 13
    screen_y = FloatingPointToFixedPoint(A=0.5, fl=13) - SUM[1]
    screen_y = screen_y * FloatingPointToFixedPoint(A=720, fl=0)
    screen_y = quantized_fl(screen_y, 0)

    res.append([screen_x, screen_y, depth])
  return res

## Rasterization

In [15]:
def div_pipe(A,At):  #not finish yet

  temp_store_1 = [0,0,0,0]
  temp_store_2 = [0,0,0,0]
  temp_store_3 = [0,0,0,0]
  temp_store_4 = [0,0,0,0]
  temp_store_5 = [0,0,0,0]

  temp_stores = [
    [0, 0, 0, 0, 0],  # temp_store_1
    [0, 0, 0, 0, 0],  # temp_store_2
    [0, 0, 0, 0, 0],  # temp_store_3
    [0, 0, 0, 0, 0],  # temp_store_4
    [0, 0, 0, 0, 0]   # temp_store_5
  ]

  A_1_1 = copy.deepcopy(A)
  A_1_1.value = A_1_1.value << 1;

  for i in range(5):
    if(A_1_1.value > At.value):
      A_2_1 = A_1_1
      A_2_1.value = (A_2_1.value - At.value) << 1
      temp_stores[i][3] = 1
    else:
      A_2_1 = A_1_1
      A_2_1.value = A_2_1.value << 1
      temp_stores[i][3] = 0

    if(A_2_1.value > At.value):
      A_3_1 = A_2_1
      A_3_1.value = (A_3_1.value - At.value) << 1
      temp_stores[i][2] = 1
    else:
      A_3_1 = A_1_1
      A_3_1.value = A_3_1.value << 1
      temp_stores[i][2] = 0

    if(A_3_1.value > At.value):
      A_4_1 = A_3_1
      A_4_1.value = (A_4_1.value - At.value) << 1
      temp_stores[i][1] = 1
    else:
      A_4_1 = A_3_1
      A_4_1.value = A_4_1.value << 1
      temp_stores[i][1] = 0

    if(A_4_1.value > At.value):
      A_1_1 = A_4_1
      A_1_1.value = (A_1_1.value - At.value) << 1
      temp_stores[i][0] = 1
    else:
      A__1 = A_4_1
      A__1.value = A_1_1.value << 1
      temp_stores[i][0] = 0

  value_acc = 0
  for j in range(5):
    value_acc = value_acc + temp_stores[j][3]*(2**(20-(4*j+1)))
    value_acc = value_acc + temp_stores[j][2]*(2**(20-(4*j+2)))
    value_acc = value_acc + temp_stores[j][1]*(2**(20-(4*j+3)))
    value_acc = value_acc + temp_stores[j][0]*(2**(20-(4*j+4)))


  new_fixed = FixedPoint(fl=20, value=value_acc)

  return new_fixed




In [16]:
def InTriangle(x1,y1,x2,y2,x3,y3,x,y):

  Ax = x1
  Ay = y1
  Bx = x2
  By = y2
  Cx = x3
  Cy = y3
  Px = x
  Py = y

  temp1 = (Bx - Ax)*(Py - Ay)
  temp2 = (By - Ay)*(Px - Ax)
  temp3 = (Cx - Bx)*(Py - By)
  temp4 = (Cy - By)*(Px - Bx)
  temp5 = (Ax - Cx)*(Py - Cy)
  temp6 = (Ay - Cy)*(Px - Cx)

  cross1 = temp1 - temp2
  cross2 = temp3 - temp4
  cross3 = temp5 - temp6

  has_neg = ((cross1.value < 0) or (cross2.value < 0) or (cross3.value < 0))
  has_pos = ((cross1.value > 0) or (cross2.value > 0) or (cross3.value > 0))

  in_triangle = not (has_neg and has_pos)

  return in_triangle


In [17]:
def GetDepth(position1,position2,position3,position1_depth,position2_depth,position3_depth,x_t,y_t):
  x1 = copy.deepcopy(position1[0])
  x2 = copy.deepcopy(position2[0])
  x3 = copy.deepcopy(position3[0])

  y1 = copy.deepcopy(position1[1])
  y2 = copy.deepcopy(position2[1])
  y3 = copy.deepcopy(position3[1])

  x = copy.deepcopy(x_t)
  y = copy.deepcopy(y_t)

  #print(f"inside:{position1_depth}")
  #print(f"inside:{position2_depth}")
  #print(f"inside:{position3_depth}")

  ######################################################

  temp0 = (x*(y2-y3))
  temp1 = (x2*(y3-y))
  temp2 = (x3*(y-y2))
  temp3 = (x1*(y-y3))
  temp4 = (x*(y3-y1))
  temp5 = (x3*(y1-y))
  temp6 = (x1*(y2-y))
  temp7 = (x2*(y-y1))
  temp8 = (x*(y1-y2))
  temp9 = (x1*(y2-y3))
  temp10 = (x2*(y3-y1))
  temp11 = (x3*(y1-y2))

  condition1 = ( (x1.value == x2.value) and (y1.value == y2.value) )
  condition2 = ( (x1.value == x3.value) and (y1.value == y3.value) )
  condition3 = ( (x2.value == x3.value) and (y2.value == y3.value) )

  if(condition1 or condition2 or condition3):
      detect = 1
  else:
      detect = 0

  overlap_v1 = ( (x.value == x1.value) and (y.value == y1.value) )
  overlap_v2 = ( (x.value == x2.value) and (y.value == y2.value) )
  overlap_v3 = ( (x.value == x3.value) and (y.value == y3.value) )

  #print(f"overlap_v1:{overlap_v1}")
  #print(f"(x1,y1)={x1},{y1}")
  #print(f"(x,y)={x},{y}")

  in_triangle = InTriangle(x1,y1,x2,y2,x3,y3,x,y)

  tempA1 = temp0 + temp1 + temp2
  tempA2 = temp3 + temp4 + temp5
  tempA3 = temp6 + temp7 + temp8
  tempAt = temp9 + temp10 + temp11

  A1 = tempA1
  A2 = tempA2
  A3 = tempA3
  At = tempAt

  if(A1.value < 0):
      A1.value = -A1.value

  if(A2.value < 0):
      A2.value = -A2.value

  if(A3.value < 0):
      A3.value = -A3.value

  if(At.value < 0):
      At.value = -At.value

  #if(in_triangle==1):
  #  print(f"A1:{A1},A2:{A2},A3:{A3},At:{At}")

  #if(x.value==663 and y.value==187):
  #    print(f"A1:{A1}")
  #    print(f"A2:{A2}")
  #    print(f"A3:{A3}")
  #    print(f"At:{At}")
  #    print(f"temp0: {temp0}")
  #    print(f"temp1: {temp1}")
  #    print(f"temp2: {temp2}")
  #    print(f"temp3: {temp3}")
  #    print(f"temp4: {temp4}")
  #    print(f"temp5: {temp5}")
  #    print(f"temp6: {temp6}")
  #    print(f"temp7: {temp7}")
  #    print(f"temp8: {temp8}")
  #    print(f"temp9: {temp9}")
  #    print(f"temp10: {temp10}")
  #    print(f"temp11: {temp11}")

  L1 = div_pipe(A=A1,At=At)
  L2 = div_pipe(A=A2,At=At)
  L3 = div_pipe(A=A3,At=At)

  #if(x.value==663 and y.value==187):
  #    print(f"DEPTH L1:{L1}")
  #    print(f"DEPTH L2:{L2}")
  #    print(f"DEPTH L3:{L3}")

  #if(in_triangle==1):
  #  print(f"L1:{L1},L2:{L2},L3:{L3}")

  temp_p1_depth = L1*position1_depth
  temp_p2_depth = L2*position2_depth
  temp_p3_depth = L3*position3_depth

  temp_p1_depth_q = quantized_fl(A=temp_p1_depth,after_quantized_fl=24)
  temp_p2_depth_q = quantized_fl(A=temp_p2_depth,after_quantized_fl=24)
  temp_p3_depth_q = quantized_fl(A=temp_p3_depth,after_quantized_fl=24)

  p1_depth = temp_p1_depth_q
  p2_depth = temp_p2_depth_q
  p3_depth = temp_p3_depth_q

  temp_current_depth = p1_depth + p2_depth + p3_depth

  temp_current_depth_q = quantized_fl(A=temp_current_depth,after_quantized_fl=20)

  if(detect==1):
    not_draw = 1
    current_depth = FixedPoint(fl=0, value=0)
  elif(overlap_v1):
    not_draw = 0
    current_depth = position1_depth
  elif(overlap_v2):
    not_draw = 0
    current_depth = position2_depth
  elif(overlap_v3):
    not_draw = 0
    current_depth = position3_depth
  else:
    not_draw = 0
    current_depth = temp_current_depth_q

  return not_draw, current_depth, in_triangle

In [18]:
def GetColor( position1, position2, position3, position1_color, position2_color, position3_color, x_t, y_t):
  x1 = copy.deepcopy(position1[0])
  x2 = copy.deepcopy(position2[0])
  x3 = copy.deepcopy(position3[0])

  y1 = copy.deepcopy(position1[1])
  y2 = copy.deepcopy(position2[1])
  y3 = copy.deepcopy(position3[1])

  x = copy.deepcopy(x_t)
  y = copy.deepcopy(y_t)

  ######################################################

  temp0 = (x*(y2-y3))
  temp1 = (x2*(y3-y))
  temp2 = (x3*(y-y2))
  temp3 = (x1*(y-y3))
  temp4 = (x*(y3-y1))
  temp5 = (x3*(y1-y))
  temp6 = (x1*(y2-y))
  temp7 = (x2*(y-y1))
  temp8 = (x*(y1-y2))
  temp9 = (x1*(y2-y3))
  temp10 = (x2*(y3-y1))
  temp11 = (x3*(y1-y2))

  condition1 = ( (x1.value == x2.value) and (y1.value == y2.value) )
  condition2 = ( (x1.value == x3.value) and (y1.value == y3.value) )
  condition3 = ( (x2.value == x3.value) and (y2.value == y3.value) )

  if(condition1 or condition2 or condition3):
      detect = 1
  else:
      detect = 0

  overlap_v1 = ( (x.value == x1.value) and (y.value == y1.value) )
  overlap_v2 = ( (x.value == x2.value) and (y.value == y2.value) )
  overlap_v3 = ( (x.value == x3.value) and (y.value == y3.value) )

  tempA1 = temp0 + temp1 + temp2
  tempA2 = temp3 + temp4 + temp5
  tempA3 = temp6 + temp7 + temp8
  tempAt = temp9 + temp10 + temp11

  A1 = tempA1
  A2 = tempA2
  A3 = tempA3
  At = tempAt

  if(A1.value < 0):
      A1.value = -A1.value

  if(A2.value < 0):
      A2.value = -A2.value

  if(A3.value < 0):
      A3.value = -A3.value

  if(At.value < 0):
      At.value = -At.value

  L1 = div_pipe(A=A1,At=At)
  L2 = div_pipe(A=A2,At=At)
  L3 = div_pipe(A=A3,At=At)

  #if(x.value==663 and y.value==187):
  #    print(f"COLOR L1:{L1}")
  #    print(f"COLOR L2:{L2}")
  #    print(f"COLOR L3:{L3}")
  #    print(f"A1:{A1},A2:{A2},A3:{A3},At:{At}")

  temp_p1_color_R = L1*position1_color[0]
  temp_p2_color_R = L2*position2_color[0]
  temp_p3_color_R = L3*position3_color[0]

  temp_p1_color_G = L1*position1_color[1]
  temp_p2_color_G = L2*position2_color[1]
  temp_p3_color_G = L3*position3_color[1]

  temp_p1_color_B = L1*position1_color[2]
  temp_p2_color_B = L2*position2_color[2]
  temp_p3_color_B = L3*position3_color[2]

  temp_p1_color_R_q = quantized_fl(A=temp_p1_color_R,after_quantized_fl=4)
  temp_p2_color_R_q = quantized_fl(A=temp_p2_color_R,after_quantized_fl=4)
  temp_p3_color_R_q = quantized_fl(A=temp_p3_color_R,after_quantized_fl=4)

  temp_p1_color_G_q = quantized_fl(A=temp_p1_color_G,after_quantized_fl=4)
  temp_p2_color_G_q = quantized_fl(A=temp_p2_color_G,after_quantized_fl=4)
  temp_p3_color_G_q = quantized_fl(A=temp_p3_color_G,after_quantized_fl=4)

  temp_p1_color_B_q = quantized_fl(A=temp_p1_color_B,after_quantized_fl=4)
  temp_p2_color_B_q = quantized_fl(A=temp_p2_color_B,after_quantized_fl=4)
  temp_p3_color_B_q = quantized_fl(A=temp_p3_color_B,after_quantized_fl=4)

  p1_color_R = temp_p1_color_R_q
  p2_color_R = temp_p2_color_R_q
  p3_color_R = temp_p3_color_R_q

  p1_color_G = temp_p1_color_G_q
  p2_color_G = temp_p2_color_G_q
  p3_color_G = temp_p3_color_G_q

  p1_color_B = temp_p1_color_B_q
  p2_color_B = temp_p2_color_B_q
  p3_color_B = temp_p3_color_B_q

  temp_current_color_R = p1_color_R + p2_color_R + p3_color_R
  temp_current_color_G = p1_color_G + p2_color_G + p3_color_G
  temp_current_color_B = p1_color_B + p2_color_B + p3_color_B

  temp_current_color_R_q = quantized_fl(A=temp_current_color_R,after_quantized_fl=0)
  temp_current_color_G_q = quantized_fl(A=temp_current_color_G,after_quantized_fl=0)
  temp_current_color_B_q = quantized_fl(A=temp_current_color_B,after_quantized_fl=0)

  if(detect==1):
    not_draw = 1
    current_color = np.array([FixedPoint(fl=0, value=0),FixedPoint(fl=0, value=0),FixedPoint(fl=0, value=0)])
  elif(overlap_v1):
    not_draw = 0
    current_color = position1_color
  elif(overlap_v2):
    not_draw = 0
    current_color = position2_color
  elif(overlap_v3):
    not_draw = 0
    current_color = position3_color
  else:
    not_draw = 0
    current_color = np.array([temp_current_color_R_q,temp_current_color_G_q,temp_current_color_B_q])

  #position1_color: np.array([R,G,B])
  return not_draw, current_color

## Test

In [19]:
import os
# dir_path = os.path.splitext(filepath)[0]
# os.makedirs(dir_path, exist_ok=True)

def create_screen(screen_buffer,i):
  print(f'Creating screen_space_{i}.dat ...')
  # assert os.path.exists(os.path.join(dir_path, "screen_space.dat")) == False, f'file already exist'
  file_name = f"screen_space_{i}.dat"
  with open(os.path.join(dir_path, file_name), "wb") as f:
    for y in range(0, 720, 4):
      for x in range(0, 1280, 4):
        for row in range(4):
          for col in range(4):
            for i in range(3):
              byte_data = f'{screen_buffer[ y+row ][ x+col ][i]:08b}'.encode('utf-8')
              f.write(byte_data)
            if row == 3 and col == 3:
              ending = '\n'.encode('utf-8')
            else:
              ending = '_'.encode('utf-8')
            f.write(ending)

In [20]:
from PIL import Image
from tqdm import tqdm
import math
from tqdm import tqdm
import imageio
import numpy as np
import matplotlib.pyplot as plt
import os

mesh_eye = { 'bunny':    [ [6.738, 1.147, 6.321], [5.239, 4.517, 1.538] ],\
             'teapot':   [ [5.124, 1.028, 4.963], [2.783, 6.167, 6.528] ] }
mesh = Mesh()

for mesh_name in mesh_eye:
    for i in [0,1]:
        position = mesh_eye[mesh_name][i]
        print(f"Rendering the {i} th shot of {mesh_name}")
                
        filepath = f'mesh/{mesh_name}.ply'
        mesh.load_ply( filepath )

        camera = Camera( eyeX=position[0], eyeY=position[1], eyeZ=position[2],
                  centerX=0, centerY=0, centerZ=0,
                  upX=0, upY=1, upZ=0,
                  screen_W=1280, screen_H=720,
                  near_clipping_plane=0.1, far_clipping_plane=10,
                  fov=45 )
        
        print("EYE:")
        print(f'(x, y, z): {camera.eyeX.value}, {camera.eyeY.value}, {camera.eyeZ.value}')
        print("number of faces:", len(mesh.faces))

        ff = 0

        for face in mesh.faces:
          # print(face)
          vertices_in_face1 = np.array([mesh.vertices[face[0]]])
          vertices_in_face2 = np.array([mesh.vertices[face[1]]])
          vertices_in_face3 = np.array([mesh.vertices[face[2]]])

          # print("\n vertices")
          # print(vertices_in_face1, '\n')
          # print(vertices_in_face2, '\n')
          # print(vertices_in_face3, '\n')

          temp1 = vertex_shader(camera=camera,vertices_in_face=vertices_in_face1)
          temp2 = vertex_shader(camera=camera,vertices_in_face=vertices_in_face2)
          temp3 = vertex_shader(camera=camera,vertices_in_face=vertices_in_face3)


          screenx1, screeny1, depth1 = temp1[0]
          screenx2, screeny2, depth2 = temp2[0]
          screenx3, screeny3, depth3 = temp3[0]

          P1 = [screenx1,screeny1]
          P2 = [screenx2,screeny2]
          P3 = [screenx3,screeny3]
          P1_depth = depth1
          P2_depth = depth2
          P3_depth = depth3

          # print(f"(x1,y1):{P1[0].value}, {P1[1].value}")
          # print(f"(x2,y2):{P2[0].value}, {P2[1].value}")
          # print(f"(x3,y3):{P3[0].value}, {P3[1].value}")
          # print(f"x1_depth:{P1_depth.value}")
          # print(f"x2_depth:{P2_depth.value}")
          # print(f"x3_depth:{P3_depth.value}")

          # print("Face vertices indices:", face)

          P1_color = mesh.colors[face[0]]
          P2_color = mesh.colors[face[1]]
          P3_color = mesh.colors[face[2]]

          # print(f"P1_color:{[P1_color[i].value for i in range(3) ]}")
          # print(f"P2_color:{[P2_color[i].value for i in range(3) ]}")
          # print(f"P3_color:{[P3_color[i].value for i in range(3) ]}")

          Ax = P1[0].value
          Bx = P2[0].value
          Cx = P3[0].value

          Ay = P1[1].value
          By = P2[1].value
          Cy = P3[1].value

          if(Ax >= Bx and Ax >= Cx):
            temp_end_x = Ax
          elif(Bx >= Ax and Bx >= Cx):
            temp_end_x = Bx
          elif(Cx >= Ax and Cx >= Bx):
            temp_end_x = Cx
          else:
            print("something wrong, find Ax Bx Cx max")

          if(Cx <= Ax and Cx <= Bx):
            temp_start_x = Cx
          elif(Bx <= Ax and Bx <= Cx):
            temp_start_x = Bx
          elif(Ax <= Bx and Ax <= Cx):
            temp_start_x = Ax
          else:
            print("something wrong, find Ax Bx Cx min")

          if(Ay >= By and Ay >= Cy):
            temp_end_y = Ay
          elif(By >= Ay and By >= Cy):
            temp_end_y = By
          elif(Cy >= Ay and Cy >= By):
            temp_end_y = Cy
          else:
            print("something wrong, find Ay By Cy max")

          if(Cy <= Ay and Cy <= By):
            temp_start_y = Cy
          elif(By <= Ay and By <= Cy):
            temp_start_y = By
          elif(Ay <= By and Ay <= Cy):
            temp_start_y = Ay
          else:
            print("something wrong, find Ay By Cy min")

          start_x = (temp_start_x//4)*4
          start_y = (temp_start_y//4)*4
          end_x = (temp_end_x//4)*4
          end_y = (temp_end_y//4)*4

          # print(f"start_x:{start_x}")
          # print(f"start_y:{start_y}")
          # print(f"end_x:{end_x}")
          # print(f"end_y:{end_y}")

          current_x = start_x
          current_y = start_y

          while (1):
            use_x = current_x
            use_y = current_y
            flag = 0

            if(((current_x + 4 > end_x) and (current_y + 4 > end_y))):
              flag=1
              current_x = current_x
              current_y = current_y
            elif( (current_x + 4 > end_x) and not(current_y + 4 > end_y) ):
              current_x = start_x
              current_y = current_y + 4
            else:
              current_x = current_x + 4
              current_y = current_y

            for k in range(16):
              if k==15:
                in_x = use_x
                in_y = use_y

              if k==14:
                in_x = use_x+1
                in_y = use_y

              if k==13:
                in_x = use_x+2
                in_y = use_y

              if k==12:
                in_x = use_x+3
                in_y = use_y

              if k==11:
                in_x = use_x
                in_y = use_y+1

              if k==10:
                in_x = use_x+1
                in_y = use_y+1

              if k==9:
                in_x = use_x+2
                in_y = use_y+1

              if k==8:
                in_x = use_x+3
                in_y = use_y+1

              if k==7:
                in_x = use_x
                in_y = use_y+2

              if k==6:
                in_x = use_x+1
                in_y = use_y+2

              if k==5:
                in_x = use_x+2
                in_y = use_y+2

              if k==4:
                in_x = use_x+3
                in_y = use_y+2

              if k==3:
                in_x = use_x
                in_y = use_y+3

              if k==2:
                in_x = use_x+1
                in_y = use_y+3

              if k==1:
                in_x = use_x+2
                in_y = use_y+3

              if k==0:
                in_x = use_x+3
                in_y = use_y+3

              in_x_f = FixedPoint(fl=0, value=in_x)
              in_y_f = FixedPoint(fl=0, value=in_y)

              not_draw_1, depth, in_triangle = GetDepth(P1, P2, P3, P1_depth, P2_depth, P3_depth, in_x_f, in_y_f)
              not_draw_2, C = GetColor(P1, P2, P3, P1_color, P2_color, P3_color, in_x_f, in_y_f)

              # if(not_draw_1 == 1):
              #   sys.exit()


              #print("")
              #print(depth)
              #print("")
              #print(C)

              #########check##########

              # in_triangle_f = is_point_in_triangle_cross_product( [P1[0].value,P1[1].value], [P2[0].value,P2[1].value], [P3[0].value,P3[1].value], [in_x, in_y] )

              # if(in_triangle != in_triangle_f):
              #   print(f"mistake: current_x:{in_x},current_y:{in_y},floating point version:{in_triangle_f}, fixed point version:{in_triangle}")

              # if(depth.value*2**(-20)>=1 and in_triangle==1 ):
              #   print(f"depth out of bound: current_x:{in_x},current_y:{in_y}, depth:{depth.value*2**(-20)}")
              #   #sys.exit()

              #print(face)
              #print(f"in_x:{in_x}, in_y:{in_y}")
              #print(f"not_draw:{not_draw_1}, depth_after:{depth.value}, in_triangle:{in_triangle}")
              #print(f"depth_org:{(camera.screen_depth_buffer[in_x+in_y*1280]).value}")
              #print(f"R:{C[0].value}")
              #print(f"G:{C[1].value}")
              #print(f"B:{C[2].value}")
              #print("")
              #print("####################################################################################")
              #print("")

              #print(f"k:{k}")

              #if(ff==1000):


              # with open("temp.txt", "a") as file:
              #   file.write(f"{face}\n")
              #   file.write(f"k:{k}\n")
              #   file.write(f"in_x:{in_x}, in_y:{in_y}\n")
              #   file.write(f"not_draw:{not_draw_1}, depth_after:{depth.value}, in_triangle:{in_triangle}\n")
              #   file.write(f"depth_org:{(camera.screen_depth_buffer[in_x+in_y*1280]).value}\n")
              #   file.write(f"R:{C[0].value}\n")
              #   file.write(f"G:{C[1].value}\n")
              #   file.write(f"B:{C[2].value}\n")
              #   file.write("\n")
              #   file.write("####################################################################################\n")
              #   file.write("\n")

              ########################

              if(not_draw_1 != not_draw_2):
                print(f"something wrong: not draw part, not_draw_1:{not_draw_1},not_draw_2:{not_draw_2}")

              if(depth.value <= (camera.screen_depth_buffer[in_x+in_y*1280]).value and in_triangle==1 and not_draw_1==0 and not_draw_2==0):
                camera.screen_depth_buffer[in_x+in_y*1280] = depth
                camera.screen_buffer[in_y][in_x] = np.array([C[0].value,C[1].value,C[2].value])

                # if(depth.value <= 0):
                #     print(f"depth:{depth.value}")

                # if(C[0].value==50 and C[1].value==128 and C[2].value==80):
                #   print(f"ff:{ff},in_x:{in_x},in_y:{in_y}")

                if(C[0].value>255 or C[0].value<0):
                  print("Color R out of bound")
                if(C[1].value>255 or C[1].value<0):
                  print("Color G out of bound")
                if(C[2].value>255 or C[2].value<0):
                  print("Color B out of bound")

            if(flag==1):
              #print(f"finish this face")
              #create_screen(camera.screen_buffer,ff)
              # if(ff%10 == 0):
              #     print(ff)

              # if(ff==1000):
              #   sys.exit()

              ff = ff + 1
              break

        # plt.imshow(camera.screen_buffer)
        # plt.axis('off')
        # plt.show()
        # # save the image
        dir_path = 'TP_FIND'
        plt.imsave(f'{dir_path}/{mesh_name}_{i}.png', camera.screen_buffer)

        print(f"Storing files in {dir_path}")

        print(f"Creating {mesh_name}_{i}_faces_array.dat ...")
        # assert os.path.exists(os.path.join(dir_path, "faces_array.dat")) == False, f'file already exist'
        with open(os.path.join(dir_path, f"{mesh_name}_{i}_faces_array.dat"), "wb") as f:
          for face in mesh.faces:
            byte_data = f'{face[0]:020b}_{face[1]:020b}_{face[2]:020b}\n'.encode('utf-8')
            f.write(byte_data)

        print(f'Creating {mesh_name}_{i}_vertices_position_array.dat ...')
        # assert os.path.exists(os.path.join(dir_path, "vertices_position_array.dat")) == False, f'file already exist'
        with open(os.path.join(dir_path, f"{mesh_name}_{i}_vertices_position_array.dat"), "wb") as f:
          for vertex in mesh.vertices:
            byte_data = f'{int_to_signed_binary_str(vertex[0].value, 24)}_{int_to_signed_binary_str(vertex[1].value, 24)}_{int_to_signed_binary_str(vertex[2].value, 24)}\n'.encode('utf-8')
            f.write(byte_data)

        print(f'Creating {mesh_name}_{i}_vertices_color_array.dat ...')
        # assert os.path.exists(os.path.join(dir_path, "vertices_color_array.dat")) == False, f'file already exist'
        with open(os.path.join(dir_path, f"{mesh_name}_{i}_vertices_color_array.dat"), "wb") as f:
          for color in mesh.colors:
            byte_data = f'{color[0].value:08b}_{color[1].value:08b}_{color[2].value:08b}\n'.encode('utf-8')
            f.write(byte_data)

        print(f'Creating {mesh_name}_{i}_screen_space.dat ...')
        # assert os.path.exists(os.path.join(dir_path, "screen_space.dat")) == False, f'file already exist'
        with open(os.path.join(dir_path, f"{mesh_name}_{i}_screen_space.dat"), "wb") as f:
          for y in range(0, 720, 4):
            for x in range(0, 1280, 4):
              for row in range(4):
                for col in range(4):
                  for i in range(3):
                    byte_data = f'{camera.screen_buffer[ y+row ][ x+col ][i]:08b}'.encode('utf-8')
                    f.write(byte_data)
                  if row == 3 and col == 3:
                    ending = '\n'.encode('utf-8')
                  else:
                    ending = '_'.encode('utf-8')
                  f.write(ending)

        print("Done !\n\n")

Rendering the 0 th shot of bunny
EYE:
(x, y, z): 7065305, 1202717, 6628049
number of faces: 2915


FileNotFoundError: [Errno 2] No such file or directory: 'TP_FIND/bunny_0.png'